# Predicción de Deserción de Clientes (Customer Churn)
### Análisis, modelado predictivo y segmentación de riesgo para dashboard ejecutivo

**Autor:** Juan Diego Roncancio Melo
**Dataset:** [IBM Telco Customer Churn](https://github.com/IBM/telco-customer-churn-on-icp4d) — 7,043 clientes de una empresa de telecomunicaciones

**Objetivo de negocio:** Identificar qué clientes tienen mayor probabilidad de cancelar su servicio (churn), entender los factores que más influyen en esa decisión, y generar un dataset de segmentación de riesgo listo para un dashboard ejecutivo en Power BI que priorice acciones de retención.

**Estructura del proyecto:**
1. Carga, limpieza y análisis exploratorio (EDA)
2. Feature engineering y entrenamiento de modelos (Regresión Logística y Random Forest)
3. Evaluación y selección del mejor modelo
4. Segmentación de riesgo y exportación para Power BI


## 1. Carga y limpieza de datos

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

df = pd.read_csv("data/telco_churn.csv")
print(f"Filas: {df.shape[0]}, Columnas: {df.shape[1]}")
df.head()

Filas: 7043, Columnas: 21


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


### Hallazgo de calidad de datos

La columna `TotalCharges` viene como texto (`object`), no como número. Al inspeccionarla,
11 filas tienen valor en blanco — corresponden a clientes con `tenure = 0` (recién ingresados,
aún sin facturación acumulada). No es un dato faltante real: se imputa como
`TotalCharges = MonthlyCharges`, representando su primer mes de facturación.

In [2]:
blanks = df[df["TotalCharges"].str.strip() == ""]
print(f"Filas con TotalCharges vacío: {len(blanks)}")
blanks[["customerID", "tenure", "MonthlyCharges", "TotalCharges"]]

Filas con TotalCharges vacío: 11


,customerID,tenure,MonthlyCharges,TotalCharges
488,4472-LVYGI,0,52.55,
753,3115-CZMZD,0,20.25,
936,5709-LVOEQ,0,80.85,
1082,4367-NUYAO,0,25.75,
1340,1371-DWPAZ,0,56.05,
3331,7644-OMVMY,0,19.85,
3826,3213-VVOLG,0,25.35,
4380,2520-SGTTA,0,20.00,
5218,2923-ARZLG,0,19.70,
6670,4075-WKNIU,0,73.35,


In [3]:
df["TotalCharges"] = df["TotalCharges"].replace(" ", np.nan).astype(float)
df["TotalCharges"] = df["TotalCharges"].fillna(df["MonthlyCharges"])
df["Churn_Flag"] = (df["Churn"] == "Yes").astype(int)

df["tenure_group"] = pd.cut(
    df["tenure"], bins=[0, 12, 24, 48, 72],
    labels=["0-12 meses", "13-24 meses", "25-48 meses", "49-72 meses"]
)
print("Limpieza completa.")

Limpieza completa.


## 2. Análisis exploratorio (EDA)

**Tasa de deserción global: 26.5%** — es un dataset moderadamente desbalanceado,
algo a tener en cuenta al evaluar el modelo (accuracy sola no es suficiente métrica).

In [4]:
churn_rate = df["Churn_Flag"].mean()
print(f"Tasa de deserción global: {churn_rate:.1%}")
df["Churn"].value_counts()

Tasa de deserción global: 26.5%


Churn
No     5174
Yes    1869
Name: count, dtype: int64

### Deserción por tipo de contrato

El hallazgo más fuerte del dataset: los contratos **mes a mes** tienen 15 veces más
deserción que los contratos a 2 años.

In [5]:
df.groupby("Contract")["Churn_Flag"].agg(["mean", "count"]).round(3)

,mean,count
Contract,,
Month-to-month,0.427,3875
One year,0.113,1473
Two year,0.028,1695


### Deserción por antigüedad del cliente (tenure)

In [6]:
df.groupby("tenure_group", observed=True)["Churn_Flag"].agg(["mean", "count"]).round(3)

,mean,count
tenure_group,,
0-12 meses,0.477,2175
13-24 meses,0.287,1024
25-48 meses,0.204,1594
49-72 meses,0.095,2239


### Deserción por método de pago

El pago por **cheque electrónico** muestra casi 3 veces más deserción que otros métodos —
posible señal de un segmento de clientes con menor compromiso o de fricción en el proceso de pago.

In [7]:
df.groupby("PaymentMethod")["Churn_Flag"].agg(["mean", "count"]).round(3)

,mean,count
PaymentMethod,,
Bank transfer (automatic),0.167,1544
Credit card (automatic),0.152,1522
Electronic check,0.453,2365
Mailed check,0.191,1612


In [8]:
df.to_csv("data/telco_churn_clean.csv", index=False)
print("Dataset limpio guardado.")

Dataset limpio guardado.


## 3. Feature Engineering y modelado predictivo

Se codifican las variables categóricas (One-Hot Encoding) y binarias (Yes/No → 1/0),
y se comparan dos modelos:

- **Regresión Logística**: modelo interpretable, útil como línea base y para explicar
  coeficientes a audiencias no técnicas.
- **Random Forest**: captura relaciones no lineales entre variables, generalmente con
  mejor desempeño en este tipo de problema.

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

binary_cols = ["Partner", "Dependents", "PhoneService", "PaperlessBilling"]
for col in binary_cols:
    df[col] = (df[col] == "Yes").astype(int)
df["gender"] = (df["gender"] == "Male").astype(int)

categorical_cols = [
    "MultipleLines", "InternetService", "OnlineSecurity", "OnlineBackup",
    "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies",
    "Contract", "PaymentMethod"
]
df_model = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

feature_cols = [c for c in df_model.columns if c not in
                ["customerID", "Churn", "Churn_Flag", "tenure_group"]]
X = df_model[feature_cols]
y = df_model["Churn_Flag"]

print(f"Features: {len(feature_cols)} | Clientes: {len(X)} | Tasa churn: {y.mean():.1%}")

Features: 30 | Clientes: 7043 | Tasa churn: 26.5%


In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### Modelo 1: Regresión Logística

In [11]:
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train_scaled, y_train)
y_pred_lr = log_reg.predict(X_test_scaled)
y_proba_lr = log_reg.predict_proba(X_test_scaled)[:, 1]

print(classification_report(y_test, y_pred_lr, target_names=["No Churn", "Churn"]))
print(f"AUC-ROC: {roc_auc_score(y_test, y_proba_lr):.3f}")

              precision    recall  f1-score   support

    No Churn       0.85      0.90      0.87      1294
       Churn       0.66      0.55      0.60       467

    accuracy                           0.81      1761
   macro avg       0.75      0.73      0.74      1761
weighted avg       0.80      0.81      0.80      1761

AUC-ROC: 0.846


### Modelo 2: Random Forest

In [12]:
rf = RandomForestClassifier(
    n_estimators=200, max_depth=8, min_samples_leaf=20,
    random_state=42, class_weight="balanced"
)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_rf, target_names=["No Churn", "Churn"]))
print(f"AUC-ROC: {roc_auc_score(y_test, y_proba_rf):.3f}")

              precision    recall  f1-score   support

    No Churn       0.91      0.75      0.82      1294
       Churn       0.53      0.80      0.64       467

    accuracy                           0.76      1761
   macro avg       0.72      0.77      0.73      1761
weighted avg       0.81      0.76      0.77      1761

AUC-ROC: 0.847


**Decisión de modelo:** aunque el AUC de ambos es similar (~0.85), se elige **Random Forest**
por su mayor *recall* en la clase Churn (80% vs 55%) — para el negocio es más costoso no detectar
a un cliente que realmente se va a ir, que generar algunas falsas alarmas.

### Variables más importantes

In [13]:
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
importances.head(10)

tenure                                 0.191409
Contract_Two year                      0.117777
TotalCharges                           0.116267
InternetService_Fiber optic            0.089756
PaymentMethod_Electronic check         0.064481
MonthlyCharges                         0.062158
Contract_One year                      0.041747
OnlineSecurity_Yes                     0.038182
TechSupport_Yes                        0.030443
StreamingMovies_No internet service    0.025582
dtype: float64

Los resultados del modelo confirman lo observado en el EDA: `tenure`, tipo de contrato,
cargos totales, tener fibra óptica y pagar con cheque electrónico son los factores de mayor peso.

In [14]:
import joblib
joblib.dump(rf, "modelo_rf_churn.pkl")
joblib.dump(scaler, "scaler.pkl")
print("Modelo guardado.")

Modelo guardado.


## 4. Segmentación de riesgo y exportación para Power BI

Se genera la probabilidad de churn para **todos** los clientes (no solo el set de prueba),
y se crean 3 segmentos de riesgo accionables para el equipo de retención:

- **Alto** (probabilidad ≥ 60%)
- **Medio** (30%–60%)
- **Bajo** (< 30%)

También se calcula el ingreso mensual concentrado en clientes de alto riesgo, para darle
una lectura de impacto financiero al hallazgo, no solo estadística.

In [15]:
def segmentar_riesgo(p):
    if p >= 0.60:
        return "Alto"
    elif p >= 0.30:
        return "Medio"
    else:
        return "Bajo"

df["Probabilidad_Churn"] = rf.predict_proba(X)[:, 1]
df["Segmento_Riesgo"] = df["Probabilidad_Churn"].apply(segmentar_riesgo)
df["Ingreso_Mensual_En_Riesgo"] = np.where(
    df["Segmento_Riesgo"] == "Alto", df["MonthlyCharges"], 0
)

resumen = df.groupby("Segmento_Riesgo").agg(
    clientes=("customerID", "count"),
    churn_real=("Churn", lambda x: (x == "Yes").mean()),
    prob_promedio=("Probabilidad_Churn", "mean")
).round(3)
resumen

,clientes,churn_real,prob_promedio
Segmento_Riesgo,,,
Alto,2030,0.625,0.759
Bajo,2969,0.039,0.136
Medio,2044,0.237,0.456


### Validación del modelo por segmento

La tasa de churn **real** crece de forma monótona entre segmentos (Bajo → Medio → Alto),
confirmando que la segmentación es confiable para priorizar acciones de retención, no solo
una probabilidad abstracta.

In [16]:
print(f"Ingreso mensual total en riesgo alto: \${df[df['Segmento_Riesgo']=='Alto']['MonthlyCharges'].sum():,.0f}")

cols_powerbi = [
    "customerID", "gender", "SeniorCitizen", "Partner", "Dependents",
    "tenure", "tenure_group", "Contract", "PaymentMethod", "InternetService",
    "MonthlyCharges", "TotalCharges", "Churn",
    "Probabilidad_Churn", "Segmento_Riesgo", "Ingreso_Mensual_En_Riesgo"
]
df_final = df[cols_powerbi].copy()
df_final["Probabilidad_Churn"] = df_final["Probabilidad_Churn"].round(4)
df_final.to_csv("churn_dashboard_dataset.csv", index=False, encoding="utf-8-sig")
print("Archivo listo para Power BI: churn_dashboard_dataset.csv")

Ingreso mensual total en riesgo alto: \$156,359
Archivo listo para Power BI: churn_dashboard_dataset.csv


<>:1: SyntaxWarning: invalid escape sequence '\$'
<>:1: SyntaxWarning: invalid escape sequence '\$'
/tmp/ipykernel_569/4157753326.py:1: SyntaxWarning: invalid escape sequence '\$'
  print(f"Ingreso mensual total en riesgo alto: \${df[df['Segmento_Riesgo']=='Alto']['MonthlyCharges'].sum():,.0f}")


## Siguiente paso: Dashboard Ejecutivo en Power BI

El archivo `churn_dashboard_dataset.csv` generado aquí se conecta directamente en Power BI
Desktop para construir un dashboard ejecutivo de retención. Ver `POWERBI_GUIDE.md` en este
repositorio para las instrucciones paso a paso, visuales recomendados y medidas DAX.

## Conclusiones de negocio

1. El contrato mes a mes es, por lejos, el mayor predictor de deserción — sugiere que
   incentivar la migración a contratos anuales es la palanca de retención más directa.
2. El riesgo se concentra en clientes nuevos (0-12 meses) — el onboarding es un momento
   crítico que merece atención proactiva.
3. El método de pago (cheque electrónico) es una señal de riesgo fácil de monitorear
   sin necesidad de encuestas ni análisis complejos adicionales.
4. El modelo permite priorizar: en vez de tratar a todos los clientes igual, el equipo
   de retención puede enfocar esfuerzos en el 29% que concentra el riesgo real (segmentos
   Alto + Medio), optimizando tiempo y presupuesto de campañas de fidelización.
